In [ ]:
!pip -q install timm grad-cam

In [ ]:
pip install pandas

In [ ]:
import os
import gc
import cv2
import math
import time
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

In [ ]:
class CFG:
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Example folder structure in Drive:
    # dog_project/
    #   train.csv
    #   test.csv
    #   train/   -> train images
    #   test/    -> test images
    ROOT = "/content/drive/MyDrive"

    TRAIN_CSV = os.path.join(ROOT, "train.csv")
    TEST_CSV = os.path.join(ROOT, "test.csv")
    TRAIN_DIR = os.path.join(ROOT, "train")
    TEST_DIR = os.path.join(ROOT, "test")

    MODEL_DIR = os.path.join(ROOT, "saved_models")
    OUTPUT_DIR = os.path.join(ROOT, "outputs")

    # RAM-safe settings
    IMG_SIZE = 160
    BATCH_SIZE = 64
    NUM_WORKERS = 2
    PIN_MEMORY = True

    N_FOLDS = 5
    EPOCHS = 5
    LR = 1e-4
    WEIGHT_DECAY = 1e-4

    TARGET_COL = "Pawpularity"
    ID_COL = "Id"

    META_COLS = [
        "Subject Focus", "Eyes", "Face", "Near", "Action", "Accessory",
        "Group", "Collage", "Human", "Occlusion", "Info", "Blur"
    ]

os.makedirs(CFG.MODEL_DIR, exist_ok=True)
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

print("Device:", CFG.DEVICE)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CFG.SEED)
torch.backends.cudnn.benchmark = True

In [ ]:
train_df = pd.read_csv(CFG.TRAIN_CSV)
test_df = pd.read_csv(CFG.TEST_CSV)

train_df["image_path"] = train_df[CFG.ID_COL].apply(lambda x: os.path.join(CFG.TRAIN_DIR, f"{x}.jpg"))
test_df["image_path"] = test_df[CFG.ID_COL].apply(lambda x: os.path.join(CFG.TEST_DIR, f"{x}.jpg"))

train_df[CFG.TARGET_COL] = train_df[CFG.TARGET_COL].astype(float).clip(0, 100)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
display(train_df.head())
display(test_df.head())

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(train_df[CFG.TARGET_COL], bins=20)
plt.title("Pawpularity Distribution")
plt.xlabel("Pawpularity")
plt.ylabel("Count")
plt.show()

# Metadata correlations
corrs = train_df[CFG.META_COLS + [CFG.TARGET_COL]].corr()[CFG.TARGET_COL].sort_values(ascending=False)
print("Correlation with Pawpularity:")
print(corrs)

In [ ]:
train_df["target_bin"] = pd.cut(
    train_df[CFG.TARGET_COL],
    bins=10,
    labels=False,
    include_lowest=True
)

skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
train_df["fold"] = -1

for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["target_bin"])):
    train_df.loc[val_idx, "fold"] = fold

print(train_df["fold"].value_counts().sort_index())

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

valid_tfms = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
class DogPopularityDataset(Dataset):
    def __init__(self, dataframe, transform=None, is_test=False):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        meta = torch.tensor(row[CFG.META_COLS].values.astype(np.float32), dtype=torch.float32)

        if self.is_test:
            return image, meta, row[CFG.ID_COL]

        target = torch.tensor(row[CFG.TARGET_COL] / 100.0, dtype=torch.float32)
        return image, meta, target

In [ ]:
class DogPopularityModel(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        self.meta_net = nn.Sequential(
            nn.Linear(len(CFG.META_COLS), 16),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.head = nn.Sequential(
            nn.Linear(in_features + 16, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, image, meta):
        img_feat = self.backbone(image)
        meta_feat = self.meta_net(meta)
        feat = torch.cat([img_feat, meta_feat], dim=1)
        out = self.head(feat).squeeze(1)
        return out

In [ ]:
def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def postprocess_predictions(preds):
    preds = np.array(preds) * 100.0
    preds = np.clip(preds, 0, 100)
    return preds

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def mixup(images, targets, metas, alpha=0.5):
    assert images.size(0) > 1, "Mixup needs batch size > 1"
    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(images.size(0)).to(images.device)
    mixed_images = lam * images + (1 - lam) * images[rand_index]
    mixed_metas = lam * metas + (1 - lam) * metas[rand_index]
    target_a = targets
    target_b = targets[rand_index]
    return mixed_images, mixed_metas, target_a, target_b, lam

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, device):
    model.train()
    criterion = nn.MSELoss()
    running_loss = 0.0

    for images, metas, targets in tqdm(loader):
        images = images.to(device)
        metas = metas.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        if np.random.rand() < 0.5:
            mixed_images, mixed_metas, target_a, target_b, lam = mixup(
                images, targets, metas, alpha=0.5
            )
            with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                preds = model(mixed_images, mixed_metas)
                loss = lam * criterion(preds, target_a) + \
                       (1 - lam) * criterion(preds, target_b)
        else:
            with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                preds = model(images, metas)
                loss = criterion(preds, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        del images, metas, targets, preds, loss

    clear_memory()
    return running_loss / len(loader)

In [ ]:
def run_fold(fold):
    print(f"\n========== FOLD {fold} ==========")

    train_fold_df = train_df[train_df["fold"] != fold].copy().reset_index(drop=True)
    valid_fold_df = train_df[train_df["fold"] == fold].copy().reset_index(drop=True)

    train_dataset = DogPopularityDataset(train_fold_df, transform=train_tfms, is_test=False)
    valid_dataset = DogPopularityDataset(valid_fold_df, transform=valid_tfms, is_test=False)

    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=True,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=CFG.PIN_MEMORY
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=CFG.PIN_MEMORY
    )

    model = DogPopularityModel().to(CFG.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=(CFG.DEVICE == "cuda"))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)

    best_rmse = float("inf")
    best_path = os.path.join(CFG.MODEL_DIR, f"best_fold_{fold}.pth")
    fold_history_path = os.path.join(CFG.OUTPUT_DIR, f"history_fold_{fold}.csv")

    history_rows = []

    for epoch in range(CFG.EPOCHS):
        print(f"\nEpoch {epoch+1}/{CFG.EPOCHS}")
        start = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, CFG.DEVICE)
        valid_loss, valid_rmse, _, _ = valid_one_epoch(model, valid_loader, CFG.DEVICE)
        scheduler.step()

        elapsed = (time.time() - start) / 60.0
        print(f"train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_rmse={valid_rmse:.4f} | time={elapsed:.2f} min")

        history_rows.append({
            "fold": fold,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            "valid_rmse": valid_rmse
        })

        if valid_rmse < best_rmse:
            best_rmse = valid_rmse
            torch.save(model.state_dict(), best_path)
            print(f"Saved best model to {best_path}")

    pd.DataFrame(history_rows).to_csv(fold_history_path, index=False)

    del train_fold_df, valid_fold_df
    del train_dataset, valid_dataset
    del train_loader, valid_loader
    del model, optimizer, scaler, scheduler
    clear_memory()

    return best_rmse

In [ ]:
fold_scores = []

for fold in range(CFG.N_FOLDS):
    best_rmse = run_fold(fold)
    fold_scores.append(best_rmse)
    clear_memory()

history_files = [os.path.join(CFG.OUTPUT_DIR, f"history_fold_{fold}.csv") for fold in range(CFG.N_FOLDS)]
history_df = pd.concat([pd.read_csv(f) for f in history_files], ignore_index=True)
history_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "training_history.csv"), index=False)

print("\nFold RMSEs:", fold_scores)
print("Mean CV RMSE:", np.mean(fold_scores))

In [ ]:
@torch.no_grad()
def get_oof_predictions():
    oof_true = np.zeros(len(train_df), dtype=np.float32)
    oof_pred = np.zeros(len(train_df), dtype=np.float32)

    for fold in range(CFG.N_FOLDS):
        print(f"\nGenerating OOF for fold {fold}")

        valid_fold_df = train_df[train_df["fold"] == fold].copy().reset_index(drop=True)
        valid_idx = train_df.index[train_df["fold"] == fold].tolist()

        valid_dataset = DogPopularityDataset(valid_fold_df, transform=valid_tfms, is_test=False)
        valid_loader = DataLoader(
            valid_dataset,
            batch_size=CFG.BATCH_SIZE,
            shuffle=False,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=CFG.PIN_MEMORY
        )

        model = DogPopularityModel().to(CFG.DEVICE)
        model.load_state_dict(torch.load(os.path.join(CFG.MODEL_DIR, f"best_fold_{fold}.pth"), map_location=CFG.DEVICE))
        model.eval()

        fold_preds = []
        fold_targets = []

        for images, metas, targets in tqdm(valid_loader):
            images = images.to(CFG.DEVICE)
            metas = metas.to(CFG.DEVICE)

            preds = model(images, metas).detach().cpu().numpy()
            fold_preds.extend(preds)
            fold_targets.extend(targets.numpy())

            del images, metas, targets, preds

        fold_preds = postprocess_predictions(fold_preds)
        fold_targets = postprocess_predictions(fold_targets)

        oof_pred[valid_idx] = fold_preds
        oof_true[valid_idx] = fold_targets

        del valid_fold_df, valid_dataset, valid_loader, model
        clear_memory()

    return oof_true, oof_pred

oof_true, oof_pred = get_oof_predictions()
oof_rmse = rmse_score(oof_true, oof_pred)
print("OOF RMSE:", oof_rmse)

oof_df = train_df[[CFG.ID_COL, CFG.TARGET_COL] + CFG.META_COLS].copy()
oof_df["prediction"] = oof_pred
oof_df["abs_error"] = np.abs(oof_df[CFG.TARGET_COL] - oof_df["prediction"])
oof_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "oof_predictions.csv"), index=False)

In [ ]:
def train_full_model():
    dataset = DogPopularityDataset(train_df, transform=train_tfms, is_test=False)
    loader = DataLoader(
        dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=True,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=CFG.PIN_MEMORY
    )

    model = DogPopularityModel().to(CFG.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=(CFG.DEVICE == "cuda"))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)

    save_path = os.path.join(CFG.MODEL_DIR, "final_model.pth")

    for epoch in range(CFG.EPOCHS):
        print(f"\nFINAL TRAIN Epoch {epoch+1}/{CFG.EPOCHS}")
        train_loss = train_one_epoch(model, loader, optimizer, scaler, CFG.DEVICE)
        scheduler.step()
        print(f"train_loss={train_loss:.4f}")

    torch.save(model.state_dict(), save_path)
    print("Saved final model:", save_path)

    del dataset, loader, model, optimizer, scaler, scheduler
    clear_memory()

train_full_model()

In [ ]:
@torch.no_grad()
def make_submission():
    dataset = DogPopularityDataset(test_df, transform=valid_tfms, is_test=True)
    loader = DataLoader(
        dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=CFG.PIN_MEMORY
    )

    model = DogPopularityModel().to(CFG.DEVICE)
    model.load_state_dict(torch.load(os.path.join(CFG.MODEL_DIR, "final_model.pth"), map_location=CFG.DEVICE))
    model.eval()

    all_ids = []
    all_preds = []

    for images, metas, ids in tqdm(loader):
        images = images.to(CFG.DEVICE)
        metas = metas.to(CFG.DEVICE)

        preds = model(images, metas).detach().cpu().numpy()
        all_ids.extend(list(ids))
        all_preds.extend(preds)

        del images, metas, preds

    all_preds = postprocess_predictions(all_preds)

    sub = pd.DataFrame({
        "Id": all_ids,
        "Pawpularity": all_preds
    })

    del dataset, loader, model
    clear_memory()

    return sub

submission_df = make_submission()
submission_path = os.path.join(CFG.OUTPUT_DIR, "submission.csv")
submission_df.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
display(submission_df.head())

In [ ]:
meta_results = []

for col in CFG.META_COLS:
    corr = train_df[[col, CFG.TARGET_COL]].corr().iloc[0, 1]
    meta_results.append((col, corr))

meta_importance_df = pd.DataFrame(meta_results, columns=["feature", "correlation_with_target"])
meta_importance_df["abs_corr"] = meta_importance_df["correlation_with_target"].abs()
meta_importance_df = meta_importance_df.sort_values("abs_corr", ascending=False)

print("\nMetadata importance:")
display(meta_importance_df)

meta_importance_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "metadata_importance.csv"), index=False)

plt.figure(figsize=(10,5))
plt.bar(meta_importance_df["feature"], meta_importance_df["abs_corr"])
plt.xticks(rotation=45)
plt.title("Metadata Feature Importance")
plt.ylabel("Absolute Correlation with Pawpularity")
plt.show()

In [ ]:
sample_view = oof_df.sort_values("prediction", ascending=False).head(8).copy()

plt.figure(figsize=(16, 10))
for i, (_, row) in enumerate(sample_view.iterrows(), 1):
    img = Image.open(os.path.join(CFG.TRAIN_DIR, f"{row['Id']}.jpg")).convert("RGB")
    plt.subplot(2, 4, i)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"True: {row[CFG.TARGET_COL]:.1f}\nPred: {row['prediction']:.1f}")
plt.tight_layout()
plt.show()

In [ ]:
summary = {
    "mean_cv_rmse": float(np.mean(fold_scores)),
    "oof_rmse": float(oof_rmse),
    "top_metadata_features": meta_importance_df[["feature", "correlation_with_target"]].head(5).to_dict(orient="records")
}

with open(os.path.join(CFG.OUTPUT_DIR, "capstone_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Done. Everything is saved in:", CFG.OUTPUT_DIR)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/outputs/oof_predictions.csv')
print(df.shape)
print(df.columns.tolist())

In [ ]:
# Average model prediction by each metadata feature
for col in ['Subject Focus', 'Eyes', 'Face', 'Near', 'Action',
            'Accessory', 'Group', 'Collage', 'Human', 'Occlusion', 'Info', 'Blur']:
    group = df.groupby(col)[['Pawpularity', 'prediction']].mean().round(2)
    print(f"\n=== {col} ===")
    print(group)

In [ ]:
top50 = df.nlargest(50, 'prediction')
print("Top 50 highest predictions:")
print("True mean:", top50['Pawpularity'].mean().round(2))
print("Pred mean:", top50['prediction'].mean().round(2))
print("Mean error:", top50['abs_error'].mean().round(2))
print("\nMetadata rates in top 50 vs full dataset:")
for col in ['Eyes', 'Face', 'Action', 'Blur']:
    top_rate = top50[col].mean()
    full_rate = df[col].mean()
    print(f"{col}: top50={top_rate:.2%}  full={full_rate:.2%}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

TRAIN_DIR = "/content/drive/MyDrive/train"
top16 = df.nlargest(16, 'prediction').reset_index(drop=True)

plt.figure(figsize=(16, 12))
for i, row in top16.iterrows():
    img = Image.open(f"{TRAIN_DIR}/{row['Id']}.jpg").convert("RGB")
    plt.subplot(4, 4, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"True: {row['Pawpularity']:.0f}  Pred: {row['prediction']:.0f}", fontsize=8)
plt.suptitle("Top 16 Highest Model Predictions", fontsize=12)
plt.tight_layout()
plt.show()